# Mineração de Dados – Saúde Mental de Adolescentes
**Dataset:** Teen Mental Health Dataset (Kaggle)  
**Disciplina:** Mineração de Dados | Campus Birigui – IFSP  

---
## 1. Seleção e Pré-processamento de Dados

In [1]:
# Instalação e importações
import subprocess, sys

for pkg in ['pandas','numpy','scikit-learn','seaborn','matplotlib','scipy','imbalanced-learn']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
# Leitura do dataset
mental_df = pd.read_csv("Teen_Mental_Health_Dataset.csv")
mental_df.columns = mental_df.columns.str.strip()

print("=== Características da Base de Dados ===")
print(f"Amostras  : {mental_df.shape[0]}")
print(f"Atributos : {mental_df.shape[1]}")
print()
mental_df.info()

FileNotFoundError: [Errno 2] No such file or directory: 'Teen_Mental_Health_Dataset.csv'

In [ ]:
# Tipos de atributos e valores ausentes
print("=== Valores Ausentes por Coluna ===")
print(mental_df.isnull().sum())

print("\n=== Registros Duplicados ===")
print(mental_df.duplicated().sum())

print("\n=== Variáveis Categóricas ===")
for col in mental_df.select_dtypes(include='object'):
    print(f"\n{col}")
    print(mental_df[col].value_counts())

In [ ]:
# Limpeza: remoção de duplicados e valores nulos
mental_df = mental_df.drop_duplicates()
mental_df = mental_df.dropna()

# Remoção de espaços extras em strings
for col in mental_df.select_dtypes(include='object'):
    mental_df[col] = mental_df[col].str.strip()

print("Após limpeza:")
print(f"  Duplicados restantes : {mental_df.duplicated().sum()}")
print(f"  Valores nulos        : {mental_df.isnull().sum().sum()}")
print(f"  Amostras             : {mental_df.shape[0]}")

In [ ]:
# Verificação de desbalanceamento de classes
print("=== Distribuição das Classes ===")
print(mental_df['depression_label'].value_counts())
print()
print(mental_df['depression_label'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

plt.figure(figsize=(6,4))
sns.countplot(x='depression_label', data=mental_df, palette='Set2')
plt.title('Distribuição da Variável Alvo (depression_label)')
plt.xlabel('Classe')
plt.ylabel('Contagem')
plt.tight_layout()
plt.show()

**Observação sobre desbalanceamento:** Se houver diferença significativa entre classes, será aplicado SMOTE (oversampling) na etapa de classificação (Seção 6).

---
## 2. Normalização e Redução de Dados

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# Separação da variável alvo
y = mental_df['depression_label']
X = mental_df.drop('depression_label', axis=1)

# Codificação de variáveis categóricas
X_encoded = pd.get_dummies(
    X,
    columns=['gender', 'platform_usage', 'social_interaction_level'],
    drop_first=True
)

# Normalização Min-Max
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_encoded)
X_scaled = pd.DataFrame(X_scaled, columns=X_encoded.columns)

print("Normalização aplicada: Min-Max Scaler")
print(f"Shape após codificação e normalização: {X_scaled.shape}")
X_scaled.head()

In [ ]:
# PCA – 2 principais componentes
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca_2d, columns=['PC1', 'PC2'])
pca_df['depression_label'] = y.values

print("Variância explicada por cada componente:")
for i, v in enumerate(pca_2d.explained_variance_ratio_, 1):
    print(f"  PC{i}: {v*100:.2f}%")
print(f"Variância total explicada: {sum(pca_2d.explained_variance_ratio_)*100:.2f}%")

plt.figure(figsize=(9,6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='depression_label', palette='Set1', alpha=0.7)
plt.title('PCA – 2 Componentes Principais')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.2f}%)')
plt.legend(title='Classe')
plt.tight_layout()
plt.show()

In [ ]:
# Scree Plot – análise da variância acumulada
pca_all = PCA()
pca_all.fit(X_scaled)
exp_var = pca_all.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

plt.figure(figsize=(10, 5))
plt.bar(range(1, len(exp_var)+1), exp_var, alpha=0.6, color='royalblue', label='Variância Individual')
plt.plot(range(1, len(exp_var)+1), cum_var, marker='o', color='firebrick', linewidth=2, label='Variância Acumulada')
plt.axhline(y=0.80, color='gray', linestyle='--', alpha=0.7, label='Alvo 80%')
plt.xlabel('Número de Componentes Principais')
plt.ylabel('Proporção da Variância')
plt.title('Scree Plot – PCA')
plt.xticks(range(1, len(exp_var)+1))
plt.legend()
plt.tight_layout()
plt.show()

**Análise da separabilidade linear:** Avaliar visualmente se as classes se separam no espaço PCA 2D. Sobreposição alta indica que os dados **não são linearmente separáveis**, sugerindo uso de kernels não-lineares (ex.: SVM RBF).

---
## 3. Análise Descritiva – Visualização

In [ ]:
# Distribuição de frequência das variáveis numéricas
num_cols = [
    'age', 'daily_social_media_hours', 'sleep_hours',
    'screen_time_before_sleep', 'academic_performance',
    'physical_activity', 'stress_level', 'anxiety_level', 'addiction_level'
]

mental_df[num_cols].hist(figsize=(15, 10), bins=20, edgecolor='black', color='steelblue')
plt.suptitle('Distribuição de Frequência – Variáveis Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Gráficos de setores – variáveis categóricas
cat_cols = ['gender', 'platform_usage', 'social_interaction_level']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, cat_cols):
    counts = mental_df[col].value_counts()
    ax.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90)
    ax.set_title(col.replace('_', ' ').title())
plt.suptitle('Distribuição das Variáveis Categóricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots por classe
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
pairs = [('stress_level','Nível de Estresse'), ('anxiety_level','Nível de Ansiedade'),
         ('addiction_level','Nível de Vício'), ('sleep_hours','Horas de Sono')]

for ax, (col, titulo) in zip(axes.flat, pairs):
    sns.boxplot(x='depression_label', y=col, data=mental_df, palette='Set2', ax=ax)
    ax.set_title(f'{titulo} por Classe')
    ax.set_xlabel('Classe')
plt.suptitle('Análise por Classe – Boxplots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de dispersão – stress vs anxiety colorido por classe
plt.figure(figsize=(8, 5))
sns.scatterplot(data=mental_df, x='stress_level', y='anxiety_level',
                hue='depression_label', palette='Set1', alpha=0.6)
plt.title('Dispersão: Estresse × Ansiedade por Classe')
plt.tight_layout()
plt.show()

---
## 4. Análise Descritiva – Medidas Estatísticas

In [ ]:
from scipy import stats

colunas_numericas = [
    'age', 'daily_social_media_hours', 'sleep_hours',
    'screen_time_before_sleep', 'academic_performance',
    'physical_activity', 'stress_level', 'anxiety_level',
    'addiction_level', 'depression_label'
]

def medidas_tendencia_central(df, col):
    s = df[col]
    return {'Media': s.mean(), 'Mediana': s.median(),
            'Ponto Medio': (s.max()+s.min())/2, 'Moda': s.mode()[0]}

def medidas_dispersao(df, col):
    s = df[col]; m = s.mean()
    return {'Amplitude': s.max()-s.min(), 'Variancia': s.var(ddof=0),
            'Desvio Padrao': s.std(ddof=0),
            'Coef. Variacao (%)': (s.std(ddof=0)/m)*100 if m != 0 else None}

def medidas_posicao_relativa(df, col):
    s = df[col]; z = stats.zscore(s)
    return {'Menor Z-score': z.min(), 'Maior Z-score': z.max(),
            'Q1': s.quantile(0.25), 'Q2 (Mediana)': s.quantile(0.50),
            'Q3': s.quantile(0.75), 'IQR': s.quantile(0.75)-s.quantile(0.25)}

resultados = []
for col in colunas_numericas:
    r = {'Coluna': col}
    r.update(medidas_tendencia_central(mental_df, col))
    r.update(medidas_dispersao(mental_df, col))
    r.update(medidas_posicao_relativa(mental_df, col))
    resultados.append(r)

df_resumo = pd.DataFrame(resultados).round(4)
df_resumo

In [ ]:
# Medidas de associação (Covariância e Correlação de Pearson)
pares = [
    ('stress_level', 'anxiety_level'),
    ('sleep_hours', 'stress_level'),
    ('addiction_level', 'daily_social_media_hours')
]

print("=== Medidas de Associação ===")
for c1, c2 in pares:
    cov = np.cov(mental_df[c1], mental_df[c2], ddof=0)[0,1]
    cor = mental_df[c1].corr(mental_df[c2])
    print(f"\n{c1}  ×  {c2}")
    print(f"  Covariância : {cov:.4f}")
    print(f"  Correlação  : {cor:.4f}")

# Matriz de correlação
plt.figure(figsize=(12, 9))
corr_matrix = X_scaled.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()

---
## 5. Análise de Grupos (Clustering)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, homogeneity_score
from sklearn.metrics.pairwise import manhattan_distances

# Redução para 2 componentes (features numéricas somente)
num_cols_cluster = [
    'age', 'daily_social_media_hours', 'sleep_hours',
    'screen_time_before_sleep', 'academic_performance',
    'physical_activity', 'stress_level', 'anxiety_level', 'addiction_level'
]
X_num_scaled = MinMaxScaler().fit_transform(mental_df[num_cols_cluster])
pca_clust = PCA(n_components=2)
X_pca_clust = pca_clust.fit_transform(X_num_scaled)
print(f"Variância explicada (PCA 2D para clustering): {sum(pca_clust.explained_variance_ratio_)*100:.2f}%")

In [ ]:
# K-Means: variando K de 2 a 8 com gráfico Elbow + Silhueta
inertias, silhuetas = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_pca_clust)
    inertias.append(km.inertia_)
    silhuetas.append(silhouette_score(X_pca_clust, lbl))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_range), inertias, marker='o', color='steelblue')
axes[0].set_title('Método do Cotovelo (Elbow)')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inércia')
axes[1].plot(list(K_range), silhuetas, marker='s', color='firebrick')
axes[1].set_title('Coeficiente de Silhueta por K')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhueta')
plt.suptitle('Seleção do K ideal – K-Means', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Plotar agrupamentos K-Means para K = 2, 3, 4
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, k in zip(axes, [2, 3, 4]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_pca_clust)
    sns.scatterplot(x=X_pca_clust[:,0], y=X_pca_clust[:,1], hue=lbl,
                    palette='Set1', alpha=0.7, ax=ax, legend='full')
    ax.set_title(f'K-Means  K={k}'); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.suptitle('Agrupamentos K-Means (diferentes valores de K)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# K-Means com diferentes métricas de distância: Euclidiana vs Manhattan
kmeans_euc = KMeans(n_clusters=3, random_state=42, n_init=10)
lbl_euc = kmeans_euc.fit_predict(X_pca_clust)

X_manhattan = manhattan_distances(X_pca_clust)
kmeans_man = KMeans(n_clusters=3, random_state=42, n_init=10)
lbl_man = kmeans_man.fit_predict(X_manhattan)

print("--- Distância Euclidiana ---")
print(f"  Silhueta     : {silhouette_score(X_pca_clust, lbl_euc):.4f}")
print(f"  Homogeneidade: {homogeneity_score(mental_df['depression_label'], lbl_euc):.4f}")

print("\n--- Distância Manhattan ---")
print(f"  Silhueta     : {silhouette_score(X_manhattan, lbl_man, metric='precomputed'):.4f}")
print(f"  Homogeneidade: {homogeneity_score(mental_df['depression_label'], lbl_man):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=X_pca_clust[:,0], y=X_pca_clust[:,1], hue=lbl_euc, palette='Set1', ax=axes[0], alpha=0.7)
axes[0].set_title('K-Means – Euclidiana')
sns.scatterplot(x=X_pca_clust[:,0], y=X_pca_clust[:,1], hue=lbl_man, palette='Set2', ax=axes[1], alpha=0.7)
axes[1].set_title('K-Means – Manhattan')
plt.suptitle('Comparação de Métricas de Distância', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# GMM – testando K e comparando com K-Means
print("--- Silhueta por K (GMM) ---")
for k in range(2, 6):
    gmm = GaussianMixture(n_components=k, random_state=42)
    lbl = gmm.fit_predict(X_pca_clust)
    print(f"  K={k}: {silhouette_score(X_pca_clust, lbl):.4f}")

gmm_final = GaussianMixture(n_components=3, random_state=42)
lbl_gmm = gmm_final.fit_predict(X_pca_clust)

print(f"\n--- GMM K=3 ---")
print(f"  Silhueta     : {silhouette_score(X_pca_clust, lbl_gmm):.4f}")
print(f"  Homogeneidade: {homogeneity_score(mental_df['depression_label'], lbl_gmm):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=X_pca_clust[:,0], y=X_pca_clust[:,1], hue=lbl_euc, palette='Set1', ax=axes[0], alpha=0.7)
axes[0].set_title('K-Means (K=3)')
sns.scatterplot(x=X_pca_clust[:,0], y=X_pca_clust[:,1], hue=lbl_gmm, palette='Set2', ax=axes[1], alpha=0.7)
axes[1].set_title('GMM (K=3)')
plt.suptitle('Comparação K-Means vs GMM', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 6. Classificação

In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold, StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, f1_score

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False
    print("imbalanced-learn não disponível – pulando SMOTE")

# ── Preparação de dados ──────────────────────────────────────────────────────
X_cls = mental_df.drop('depression_label', axis=1)
y_cls = mental_df['depression_label']

X_enc = pd.get_dummies(X_cls, columns=['gender','platform_usage','social_interaction_level'], drop_first=True)

# Holdout 70/30 estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y_cls, test_size=0.3, random_state=42, stratify=y_cls
)

scaler_cls = MinMaxScaler()
X_train_s = scaler_cls.fit_transform(X_train)
X_test_s  = scaler_cls.transform(X_test)

# SMOTE no treino (se disponível)
if SMOTE_AVAILABLE:
    sm = SMOTE(random_state=42)
    X_res, y_res = sm.fit_resample(X_train_s, y_train)
    print("SMOTE aplicado – distribuição após balanceamento:")
    print(pd.Series(y_res).value_counts())
else:
    X_res, y_res = X_train_s, y_train

cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
print(f"\nHoldout – treino: {X_train_s.shape[0]}  teste: {X_test_s.shape[0]}")

In [ ]:
# Função auxiliar de avaliação
def avaliar(nome, modelo, X_tr, y_tr, X_te, y_te, cv):
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_te)

    # Confusion matrix
    cm = confusion_matrix(y_te, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=modelo.classes_)
    disp.plot(cmap='Blues')
    plt.title(f'Matriz de Confusão – {nome}')
    plt.tight_layout(); plt.show()

    # Métricas holdout
    acc  = accuracy_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred, average='weighted')
    print(f"\n[{nome}] Holdout → Acurácia: {acc:.4f}  |  F1-Score: {f1:.4f}")
    print(classification_report(y_te, y_pred))

    # Cross-Validation k=10
    cv_scores = cross_val_score(modelo, X_tr, y_tr, cv=cv, scoring='f1_weighted')
    print(f"[{nome}] CV-10   → F1 médio: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    return {'Modelo': nome, 'Acurácia (Holdout)': round(acc,4),
            'F1 (Holdout)': round(f1,4), 'F1 médio (CV-10)': round(cv_scores.mean(),4)}

In [ ]:
# ── Árvore de Decisão ───────────────────────────────────────────────────────
dt = DecisionTreeClassifier(random_state=42)
grid_dt = GridSearchCV(dt, {'criterion':['gini','entropy'],'max_depth':[3,5,7,10]},
                       cv=5, scoring='f1_weighted')
grid_dt.fit(X_res, y_res)
print("Melhores parâmetros DT:", grid_dt.best_params_)
res_dt = avaliar("Árvore de Decisão", grid_dt.best_estimator_, X_res, y_res, X_test_s, y_test, cv10)

In [ ]:
# ── K-NN ────────────────────────────────────────────────────────────────────
knn = KNeighborsClassifier()
grid_knn = GridSearchCV(knn, {'n_neighbors':[3,5,7,9,11],'weights':['uniform','distance']},
                        cv=5, scoring='f1_weighted')
grid_knn.fit(X_res, y_res)
print("Melhores parâmetros KNN:", grid_knn.best_params_)
print(f"Melhor K encontrado via GridSearchCV com CV=5 e métrica F1-weighted: K={grid_knn.best_params_['n_neighbors']}")
res_knn = avaliar("K-NN", grid_knn.best_estimator_, X_res, y_res, X_test_s, y_test, cv10)

In [ ]:
# ── SVM ─────────────────────────────────────────────────────────────────────
svm = SVC(random_state=42)
grid_svm = GridSearchCV(svm, {'C':[0.1,1,10],'kernel':['rbf','linear']},
                        cv=5, scoring='f1_weighted')
grid_svm.fit(X_res, y_res)
print("Melhores parâmetros SVM:", grid_svm.best_params_)
res_svm = avaliar("SVM", grid_svm.best_estimator_, X_res, y_res, X_test_s, y_test, cv10)

In [ ]:
# ── MLP (Rede Neural) ───────────────────────────────────────────────────────
# Arquitetura: 2 camadas escondidas (128 e 64 neurônios), ativação ReLU
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    max_iter=500,
    random_state=42
)
print("Arquitetura MLP: camadas=(128, 64), ativação=relu, max_iter=500")
res_mlp = avaliar("MLP", mlp, X_res, y_res, X_test_s, y_test, cv10)

In [ ]:
# ── Comparação Final ────────────────────────────────────────────────────────
df_comparacao = pd.DataFrame([res_dt, res_knn, res_svm, res_mlp])
df_comparacao = df_comparacao.sort_values('F1 (Holdout)', ascending=False).reset_index(drop=True)

print("=== COMPARAÇÃO FINAL DOS CLASSIFICADORES ===")
print(df_comparacao.to_string(index=False))

# Gráfico de comparação
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df_comparacao.plot(x='Modelo', y='Acurácia (Holdout)', kind='bar', ax=axes[0],
                   color='steelblue', legend=False)
axes[0].set_title('Acurácia – Holdout'); axes[0].set_ylim(0,1); axes[0].tick_params(axis='x', rotation=20)

df_comparacao.plot(x='Modelo', y=['F1 (Holdout)','F1 médio (CV-10)'], kind='bar', ax=axes[1], colormap='Set2')
axes[1].set_title('F1-Score: Holdout vs CV-10'); axes[1].set_ylim(0,1); axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Comparação Final dos Classificadores', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()